## Task 1: Load PDF Documents

In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('physical.pdf')

# Load PDF Document
docs = loader.load()

print("Total Pages:", len(docs))
print("Sample Page Content:\n", docs[0].page_content[:500])

Total Pages: 10
Sample Page Content:
 Physical education (PE) aims at the optimum development 
of an individual through continuous process of learning and 
participation in guided physical activities. In other words, it 
aims at optimum physical, mental and social development of 
an individual. 
What is Physical  Education ?
According to Webster’s Dictionary, “Physical education is a 
part of education which gives instructions in the development 
and care of the body rending from simple callisthenic exercises 
to a course of study p


## Task 2: Text Splitting

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Text Splitting
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)

print("Total Chunks:", len(chunks))

Total Chunks: 34


## Task 3: Create Embeddings

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

# Create Embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Embedding Dimension:", len(embeddings.embed_query("Hello world")))

c:\Users\ManishAgrahari\Desktop\my_learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2255.80it/s]


Embedding Dimension: 384


## Task 4: Vector Store Setup

In [8]:
from langchain_community.vectorstores import FAISS

# Creating Vector store of chunks after embeddings
vector_store = FAISS.from_documents(chunks, embeddings)

# Retriever from vector store 
retriever = vector_store.as_retriever()

## Task 5: RAG Prompt Template

In [9]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_groq import ChatGroq

system_prompt = '''
You are a helpful PDF question-answering assistant.
Answer using only the retrieved PDF context.
If the answer is not found in the context, say:
I don't know.

context:
{context}
'''

prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human', '{input}')
])

# Initialize the Groq LLM
llm = ChatGroq(model="openai/gpt-oss-120b",temperature=0)

## Task 6: Build Conversational RAG Chain

In [12]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Combine retrieved PDF chunks with the prompt and LLM
documents_chain = create_stuff_documents_chain(llm, prompt)

# Connect the retriever with the documents chain
retrieval_chain = create_retrieval_chain(retriever, documents_chain)

print("Conversational RAG chain created successfully!")

Conversational RAG chain created successfully!


## Task 7: Maintain Message History

In [13]:
# As in above prompt I already have MessagesPlaceholder(variable_name='chat_history') 
# Maintain Message History

chat_history = []

## Task 8: Trimming Chat History

In [15]:
from langchain_core.messages import trim_messages, HumanMessage, AIMessage

trimmer = trim_messages(
    token_counter=llm, 
    max_tokens=2000, 
    strategy='last', 
    include_system=True, 
    allow_partial=False, 
    start_on='human'
)

# TASK 7 & 8: Maintain and trim chat history

def chat(question):
    # Trim history before sending it to the chain
    trimmed_history = trimmer.invoke(chat_history)

    # using retrieval_chain to invoke query
    response = retrieval_chain.invoke({
        "input": question,
        "chat_history": trimmed_history
    })

    answer = response["answer"]

    # Save the latest user question and AI answer
    chat_history.extend([
        HumanMessage(content=question),
        AIMessage(content=answer)
    ])
    return answer

# Test the chatbot
while True:
    user_input = input('Please ask question related to pdf')

    if user_input.lower() == 'quit':
        break

    answer = chat(user_input)

    print('\nAssistant:', answer)



Assistant: Coronavirus is the newly discovered virus that causes the infectious disease Covid‑19. It was first identified during the outbreak that began in Wuhan, China, in December 2019, and is therefore called the Novel (new) Coronavirus (NCoV).


c:\Users\ManishAgrahari\Desktop\my_learning\.venv\Lib\site-packages\langchain_core\language_models\base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))



Assistant: Coronavirus spreads from an infected person to others mainly through small droplets that are expelled from the nose or mouth when the person coughs, sneezes or simply exhales. These droplets can:

* Land on nearby objects and surfaces; a person can become infected by touching those contaminated surfaces and then touching their eyes, nose or mouth.  
* Be inhaled directly by someone who is close to the infected person, especially within about 1 meter.

Because of this droplet transmission, maintaining a distance of more than 1 meter from a sick individual, practicing good hand hygiene, and avoiding touching the face after contact with potentially contaminated surfaces are important measures to prevent spread.


## Task 9: Follow-Up Q&A Testing

In [16]:
# Test 1: Initial factual question
question1 = "What is the main topic of this PDF?"
answer1 = chat(question1)

print("Test 1 - Initial Question")
print("Question:", question1)
print("Answer:", answer1)


# Test 2: Follow-up question
question2 = "Can you explain the first point in more detail?"
answer2 = chat(question2)

print("\nTest 2 - Follow-up Question")
print("Question:", question2)
print("Answer:", answer2)


# Test 3: Clarification question
question3 = "Can you give an example of that?"
answer3 = chat(question3)

print("\nTest 3 - Clarification Question")
print("Question:", question3)
print("Answer:", answer3)


Test 1 - Initial Question
Question: What is the main topic of this PDF?
Answer: The PDF is about **Health and Physical Education (HPE) for class X**, covering topics such as consumer health, health needs and services, community health, and related concepts in economics, social science, and biology.

Test 2 - Follow-up Question
Question: Can you explain the first point in more detail?
Answer: The first point listed in the PDF is **Physical fitness**.  

- **Definition** – Physical fitness is described as the condition in which a person is physically fit and has developed strong endurance, speed and strength.  
- **Importance** – It is presented as essential for leading a happy, healthy, vigorous and active life. When students achieve good physical fitness they are better able to cope with daily demands, perform well in sports and other activities, and maintain overall well‑being.  

Thus, the PDF emphasizes that building physical fitness is a primary objective of Physical Education, bec

## Task 10: Build Final Chatbot Application

In [ ]:
# Please refer app.py for task10 streamlit app

## Task 11: Observations & Insights

1. Difference between PDF Q&A and Conversational PDF Q&A

PDF Q&A answers individual questions using retrieved PDF content. Conversational PDF Q&A also maintains previous messages, allowing users to ask follow-up questions without repeating the entire context.

2. Role of message history in follow-up questions

Message history helps the LLM understand references to earlier questions and answers. MessagesPlaceholder passes the stored conversation into the prompt so the model can interpret follow-up questions correctly.

3. Trade-offs between long memory and performance

Longer conversation history provides more context but increases token usage, processing time, and cost. Shorter history improves efficiency but may remove information needed to understand earlier references.

4. How trimming history affects answer quality

Trimming removes older messages to keep the conversation within a specified budget. It improves efficiency, but important details from earlier turns may be lost, potentially reducing the quality of follow-up answers.